In [6]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from google.colab import files

# 1. Animation & Output Video Configurations
N_TERMS = 8          # Number of circles. Higher numbers make sharper edges.
FPS = 60             # High frame rate for buttery-smooth playback
DURATION_SEC = 10    # Total length of video loop in seconds
TOTAL_FRAMES = FPS * DURATION_SEC
COLOR_CIRCLE = "white" # '#38bdf8'
COLOR_RADIUS = '#ffcc00'
COLOR_WAVE = '#a855f7'


# Setup a clean dark layout tailored for presentation slides
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(12, 6.75), dpi=120) # Optimized DPI for Colab display
ax.set_xlim(-2.2, 5.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.axis('off')

# 2. Design the Visual Elements
radius_lines, = ax.plot([], [], color=COLOR_RADIUS, lw=1.2, zorder=3) # Orange/Gold lines
wave_line, = ax.plot([], [], color=COLOR_WAVE, lw=2.5, zorder=2)    # Purple trailing wave
pointer_line, = ax.plot([], [], color='white', linestyle='--', lw=1, alpha=0.5)

# Initialize the stack of circles
circles = [plt.Circle((0, 0), radius=0, color=COLOR_CIRCLE, fill=False, lw=0.8, alpha=0.4) for _ in range(N_TERMS)]
for circle in circles:
    ax.add_patch(circle)

wave_x_history = []
wave_y_history = []
WAVE_START_X = 1.3

def animate(frame):
    global wave_x_history, wave_y_history

    if frame == 0:
        wave_x_history.clear()
        wave_y_history.clear()

    t = (frame / TOTAL_FRAMES) * 2 * np.pi
    x_centers = [0.0]
    y_centers = [0.0]

    for i in range(N_TERMS):
        freq = 2 * i + 1
        radius = 4 / (np.pi * freq)

        next_x = x_centers[-1] + radius * np.cos(freq * t)
        next_y = y_centers[-1] + radius * np.sin(freq * t)

        circles[i].center = (x_centers[-1], y_centers[-1])
        circles[i].set_radius(radius)

        x_centers.append(next_x)
        y_centers.append(next_y)

    pen_x, pen_y = x_centers[-1], y_centers[-1]

    velocity = (5.5 - WAVE_START_X) / (2*TOTAL_FRAMES)
    wave_x_history = [x + velocity for x in wave_x_history]

    wave_x_history.insert(0, WAVE_START_X)
    wave_y_history.insert(0, pen_y)

    if len(wave_x_history) > 2*TOTAL_FRAMES:
        wave_x_history.pop()
        wave_y_history.pop()

    radius_lines.set_data(x_centers, y_centers)
    wave_line.set_data(wave_x_history, wave_y_history)
    pointer_line.set_data([pen_x, WAVE_START_X], [pen_y, pen_y])

    return [radius_lines, wave_line, pointer_line] + circles

# Title formula
plt.title(r"$f_N(t) = \frac{4}{\pi} \sum_{k=0}^{N-1} \frac{1}{2k+1} \sin((2k+1)t)$",
          color='#e2e8f0', fontsize=15, pad=20)
plt.tight_layout()

# 3. Compile the Animation
ani = animation.FuncAnimation(fig, animate, frames=TOTAL_FRAMES, blit=True)

# 4. Save to the Colab Cloud Directory
video_filename = "fourier_series_colab.mp4"
writer = animation.FFMpegWriter(fps=FPS, bitrate=3000)
ani.save(video_filename, writer=writer)
plt.close() # Prevents extra static plot from rendering underneath

# 5. Trigger Automatic Browser Download
files.download(video_filename)

# 6. Render an Interactive Video Player right inside your Colab Cell
HTML(ani.to_html5_video())


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>